## 1.需求分析
- 基于langchain和多模态大模型的食谱推荐应用。用户可以拍摄实物照片，AI自动识别图片中的食材，根据食物搜索相关食谱推荐给用户
- 功能特性：  
1.📸 图片识别 - 上传食材图片，自动识别其中的食材  
2.🔍 智能搜索 - 根据识别的食材搜索相关食谱  
3.🍽️ 智能排序 - 按推荐度、难度、营养价值对食谱进行排序  
4.💡 创意建议 - 当找不到合适食谱时，提供创意搭配建议    
5.💬 对话交互 - 聊天式界面，支持图片上传 + 文本对话

## 2.开发流程
1.先定义模型  
2.再定义工具  
3.再搞记忆管理

In [18]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("BASE_URL")

In [19]:
#创建模型
model = init_chat_model(
    model="qwen3.6-plus",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
)

In [22]:
#自定义联网搜索工具
from langchain_tavily import TavilySearch
from langchain.tools import tool

tavily_api_key = os.environ["TAVILY_API_KEY"]

#使用预定义搜索工具
search= TavilySearch(
    max_results=5,
    topic="general",
)

#提示词工程
system_prompt = """
    #你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
    1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
    2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
    3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
    4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

    #指令:
    请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
    
    """


#自定义工具
@tool
def web_search(query:str):
    """
    根据用户提供的食材照片或清单搜索食谱
    """
    return search.invoke(query)

In [23]:
#记忆管理
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

concetion = sqlite3.connect("Database/chief.db",check_same_thread=False)
checkpointer = SqliteSaver(concetion)
checkpointer.setup()



In [24]:
#创建Agent
from langchain.agents import create_agent

clint = create_agent(
    model = model,
    tools = [web_search],
    system_prompt = system_prompt,
    checkpointer = checkpointer

)

In [25]:
#测试
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "6"}}

response = clint.invoke({"messages": [multimodal_message]}, config)

In [26]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================ Human Message =================================

[{'type': 'image', 'url': 'https://img.freepik.com/free-photo/arrangement-different-foods-organized-fridge_23-2149099882.jpg'}, {'type': 'text', 'text': '帮我

In [27]:
#继续问
response = clint.invoke(
    {"messages": [HumanMessage(content="我喜欢第1道菜，可以说的更详细点吗？")]},
    config
)

# 友好打印
response['messages'][-1].pretty_print()

================================== Ai Message ==================================

没问题！作为您的私人厨师，我很乐意为您拆解这道**「地中海风味双拼暖沙拉（三文鱼+鸡胸肉）」**的详细操作指南。

这道菜的精髓在于：**热腾腾的煎肉/菌菇**与**清爽脆嫩的生鲜蔬菜**形成口感对比，既能满足对肉的渴望，又不会觉得油腻。

以下是保姆级的制作步骤：

---

### 🍽️ 菜名：地中海风味双拼暖沙拉
**预计耗时**：15-20分钟
**难度等级**：⭐⭐（新手友好，只要会开火就行）

---

### 🛒 第一步：食材预处理（切配与腌制）

1.  **鸡胸肉处理（防柴秘诀）**：
    *   从冰箱取出鸡胸肉，洗净擦干。
    *   **切法**：不要整块煎，容易外焦里生。建议**横向片开**成两片薄肉排，或者切成**一口大小的肉丁/条**。
    *   **腌制**：放入碗中，加少许**盐、黑胡椒**、一点点**料酒（或柠檬汁）**，抓匀腌制10分钟。*（如果有淀粉，加半勺淀粉抓匀，肉会更嫩）*

2.  **三文鱼处理**：
    *   洗净擦干水分（这一步很重要，防止煎的时候溅油）。
    *   **切法**：如果这块三文鱼比较厚，建议切成**2-3厘米宽的厚片**；如果较薄，可以整块煎。
    *   **腌制**：两面撒少许**盐和黑胡椒**即可，保留鱼肉原本的鲜甜。

3.  **蔬菜处理**：
    *   **口蘑**：洗净，切成厚片（太薄容易缩水）。
    *   **西兰花**：切成小朵。
    *   **洋葱**（右下角那个）：切1/4个即可，切成细丝或小丁。
    *   **其他蔬菜**：混合生菜洗净沥干；小番茄对半切；红彩椒去籽切条。

---

### 🔥 第二步：烹饪阶段（按顺序来，一锅到底）

1.  **焯水西兰花**：
    *   烧一锅水，水开后加一勺盐和几滴油（保持翠绿）。
    *   放入西兰花煮1-2分钟，捞出过一下凉水，沥干备用。

2.  **煎口蘑（释放鲜味）**：
    *   平底锅烧热，倒一点点油。
    *   放入切好的**口蘑片**，中火煎。口蘑会出水，不用管它，继续煎直到水